# DeepSeek Coding Agent Model Server

A lightweight experimental setup for serving **DeepSeek Coder 6.7B Instruct** from Google Colab and exposing it through a FastAPI endpoint.

### Architecture

`DeepSeek Coder 6.7B → 4-bit GPU inference → FastAPI → Cloudflare Quick Tunnel → Local Agent Harness`

> This notebook is intended for experimentation and development, not production deployment.

## 1. Install dependencies

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes sentencepiece fastapi uvicorn

## 2. Imports and environment check

The model is intended to run on a Colab GPU such as a Tesla T4.

In [ ]:
import re
import subprocess
import threading
import time

import torch

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("A CUDA-enabled Colab runtime is required.")

## 3. Model configuration

The model is loaded using **4-bit NF4 quantization** with FP16 computation to fit the 6.7B parameter model on a T4.

In [ ]:
MODEL_NAME = "deepseek-ai/deepseek-coder-6.7b-instruct"

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("Model:", MODEL_NAME)
print("Quantization: 4-bit NF4")
print("Compute dtype: float16")

## 4. Load tokenizer

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
)

print("Tokenizer loaded.")

## 5. Load the 4-bit model

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quant_config,
    device_map="auto",
    dtype=torch.float16,
    trust_remote_code=True,
)

model.eval()

print("Model loaded successfully.")

## 6. Test local generation

The instruction model is used through its chat template so that the same interaction format can be used by the API.

In [ ]:
def generate_response(prompt, max_new_tokens=128, temperature=0.2):
    messages = [
        {
            "role": "user",
            "content": prompt,
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to(model.device)

    with torch.inference_mode():
        outputs = model.generate(
            inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=temperature > 0,
            use_cache=True,
            eos_token_id=tokenizer.eos_token_id,
        )

    generated_tokens = outputs[0, inputs.shape[-1]:]

    return tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True,
    )


prompt = "Write a Python function to check whether a number is prime."
print(generate_response(prompt))

## 7. Optional inference benchmark

In [ ]:
benchmark_prompt = "Write a Python function to calculate the factorial of a number."

# Warm-up
generate_response(benchmark_prompt, max_new_tokens=32, temperature=0.0)
torch.cuda.synchronize()

start = time.perf_counter()

messages = [{"role": "user", "content": benchmark_prompt}]
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
).to(model.device)

with torch.inference_mode():
    outputs = model.generate(
        inputs,
        max_new_tokens=128,
        do_sample=False,
        use_cache=True,
        eos_token_id=tokenizer.eos_token_id,
    )

torch.cuda.synchronize()

elapsed = time.perf_counter() - start
generated_tokens = outputs.shape[1] - inputs.shape[-1]

print("Generated tokens:", generated_tokens)
print("Time:", round(elapsed, 2), "seconds")
print("Speed:", round(generated_tokens / elapsed, 2), "tokens/sec")

## 8. FastAPI server

The `/generate` endpoint accepts a prompt and generation parameters and returns the model output.

In [ ]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field


app = FastAPI(title="DeepSeek Coding Model API")


class GenerateRequest(BaseModel):
    prompt: str
    max_new_tokens: int = Field(default=256, ge=1, le=2048)
    temperature: float = Field(default=0.2, ge=0.0, le=2.0)


@app.get("/")
def root():
    return {
        "status": "online",
        "model": MODEL_NAME,
    }


@app.post("/generate")
def generate(request: GenerateRequest):
    try:
        response = generate_response(
            prompt=request.prompt,
            max_new_tokens=request.max_new_tokens,
            temperature=request.temperature,
        )

        return {"response": response}

    except Exception as exc:
        raise HTTPException(
            status_code=500,
            detail=str(exc),
        ) from exc


print("FastAPI application created.")

## 9. Start FastAPI

The server runs in a background thread so the Colab notebook remains usable.

In [ ]:
SERVER_PORT = 8000


def run_server():
    import uvicorn

    uvicorn.run(
        app,
        host="0.0.0.0",
        port=SERVER_PORT,
    )


server_thread = threading.Thread(
    target=run_server,
    daemon=True,
)
server_thread.start()

print(f"FastAPI server started on port {SERVER_PORT}.")

## 10. Install Cloudflare Tunnel

A Quick Tunnel exposes the local Colab port through a temporary `trycloudflare.com` URL.

**Note:** Quick Tunnel URLs are temporary and should not be treated as private or production endpoints.

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

## 11. Start the Cloudflare Quick Tunnel

In [ ]:
TUNNEL_TARGET = f"http://127.0.0.1:{SERVER_PORT}"

cloudflare_process = subprocess.Popen(
    [
        "cloudflared",
        "tunnel",
        "--url",
        TUNNEL_TARGET,
        "--no-autoupdate",
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

public_url = None

for _ in range(30):
    line = cloudflare_process.stdout.readline()

    if line:
        match = re.search(
            r"https://[a-zA-Z0-9.-]+\.trycloudflare\.com",
            line,
        )

        if match:
            public_url = match.group(0)
            break

    time.sleep(1)

if public_url:
    print("Public API URL:", public_url)
    print("Generate endpoint:", f"{public_url}/generate")
else:
    print("Could not automatically detect the tunnel URL.")

## 12. Test the remote endpoint

Run this after the tunnel is available. The same URL can be used by the local coding-agent harness.

In [ ]:
import requests

test_response = requests.post(
    f"{public_url}/generate",
    json={
        "prompt": "Write a Python function to check whether a number is prime.",
        "max_new_tokens": 128,
        "temperature": 0.2,
    },
    timeout=180,
)

test_response.raise_for_status()
print(test_response.json()["response"])

## Local harness connection

The local VS Code harness can call:

```text
POST {PUBLIC_URL}/generate
```

with:

```json
{
  "prompt": "...",
  "max_new_tokens": 512,
  "temperature": 0.1
}
```

This notebook only provides the remote model inference service. The agent loop, tools, workspace and session logic remain in the local VS Code harness.